In [1]:
import os
import json
import csv

# --- CONFIGURATION ---
# IMPORTANT: The Base file goes first, the Patch file goes second to overwrite!
INPUT_FILES = [
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-ec-result/3B_baseline_extreme_FULL_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-ec-result/3B_ablation_patch_600-799.json"
]

MASTER_JSON_OUTPUT = "/kaggle/working/MASTER_3B_EXTREME_Results.json"
MASTER_CSV_OUTPUT = "/kaggle/working/MASTER_3B_EXTREME_Metrics.csv"

# MVBench Task Mapping by Index (200 questions per task)
MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS):
        return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Commencing Grand Aggregation for 3B EXTREME COMPRESSION...\n")

# --- PHASE 1: MERGE AND OVERWRITE ---
master_records = {}

for file_path in INPUT_FILES:
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            try:
                data = json.load(f)
                loaded_count = 0
                patched_count = 0
                for record in data:
                    q_id = record["id"]
                    if q_id in master_records:
                        patched_count += 1
                    master_records[q_id] = record
                    loaded_count += 1
                
                print(f"[+] Loaded {loaded_count} records from {os.path.basename(file_path)} (Overwrote {patched_count} old records).")
            except json.JSONDecodeError:
                print(f"[!] ERROR: {os.path.basename(file_path)} is corrupted or empty.")
    else:
        print(f"[-] WARNING: {os.path.basename(file_path)} not found. Skipping.")

# Sort chronologically by ID to keep the final JSON perfectly ordered
sorted_records = [master_records[k] for k in sorted(master_records.keys())]

# --- PHASE 2: CALCULATE METRICS ---
task_metrics = {task: {"correct": 0, "total": 0, "ttft_sum": 0.0, "tps_sum": 0.0} for task in MVBENCH_TASKS}
overall_correct = 0
overall_ttft = 0.0
overall_tps = 0.0
peak_vram = 0.0

for r in sorted_records:
    task_name = get_task_name(r["id"])
    
    # Safely handle missing ground truth keys just in case
    gt_letter = r.get("ground_truth_letter", "")
    is_correct = (r["response"].strip().upper() == gt_letter.strip().upper())
    
    if is_correct:
        task_metrics[task_name]["correct"] += 1
        overall_correct += 1
        
    task_metrics[task_name]["total"] += 1
    task_metrics[task_name]["ttft_sum"] += r.get("ttft_seconds", 0.0)
    task_metrics[task_name]["tps_sum"] += r.get("tokens_per_second", 0.0)
    
    overall_ttft += r.get("ttft_seconds", 0.0)
    overall_tps += r.get("tokens_per_second", 0.0)
    
    if r.get("peak_vram_gb", 0.0) > peak_vram:
        peak_vram = r.get("peak_vram_gb", 0.0)

total_records = len(sorted_records)

# --- PHASE 3: CONSOLE REPORT ---
print("\n===========================================================================")
print(f"{'MVBench Sub-Task':<30} | {'Accuracy':<10} | {'Avg TTFT':<12} | {'Avg TPS'}")
print("===========================================================================")

csv_data = []
csv_headers = ["Task", "Accuracy_Percent", "Correct", "Total", "Avg_TTFT_sec", "Avg_TPS"]

for task in MVBENCH_TASKS:
    data = task_metrics[task]
    if data["total"] > 0:
        acc = (data["correct"] / data["total"]) * 100
        avg_ttft = data["ttft_sum"] / data["total"]
        avg_tps = data["tps_sum"] / data["total"]
        
        print(f"{task:<30} | {acc:>5.2f}%    | {avg_ttft:>6.2f} sec   | {avg_tps:>5.2f}")
        csv_data.append([task, round(acc, 2), data["correct"], data["total"], round(avg_ttft, 2), round(avg_tps, 2)])
    else:
        print(f"{task:<30} | NO DATA")

if total_records > 0:
    final_acc = (overall_correct / total_records) * 100
    final_avg_ttft = overall_ttft / total_records
    final_avg_tps = overall_tps / total_records

    print("===========================================================================")
    print(f"{'OVERALL ACCURACY':<30} | {final_acc:>5.2f}%    ({overall_correct}/{total_records})")
    print("===========================================================================\n")
    
    print("=============================================")
    print("       HARDWARE PERFORMANCE SUMMARY       ")
    print("=============================================")
    print(f"Average Time-to-First-Token:     {final_avg_ttft:.2f} seconds")
    print(f"Average Tokens-per-Second:       {final_avg_tps:.2f} tokens/s")
    print(f"Absolute Peak VRAM Usage:        {peak_vram:.2f} GB")
    print("=============================================\n")

# --- PHASE 4: DISK EXPORT ---
with open(MASTER_JSON_OUTPUT, "w") as f:
    json.dump(sorted_records, f, indent=4)

with open(MASTER_CSV_OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers)
    writer.writerows(csv_data)
    
    if total_records > 0:
        writer.writerow([])
        writer.writerow(["OVERALL", round(final_acc, 2), overall_correct, total_records, round(final_avg_ttft, 2), round(final_avg_tps, 2)])
        writer.writerow(["PEAK_VRAM_GB", peak_vram, "", "", "", ""])

print(f"Aggregation Complete!")
print(f"Total Unique Records: {total_records} / 4000")
print(f"Master JSON saved to: {MASTER_JSON_OUTPUT}")
print(f"Metrics CSV saved to: {MASTER_CSV_OUTPUT}")

Commencing Grand Aggregation for 3B EXTREME COMPRESSION...

[+] Loaded 3600 records from 3B_baseline_extreme_FULL_results.json (Overwrote 0 old records).
[+] Loaded 200 records from 3B_ablation_patch_600-799.json (Overwrote 0 old records).

MVBench Sub-Task               | Accuracy   | Avg TTFT     | Avg TPS
action_sequence                | 36.50%    |   0.30 sec   | 76.25
moving_count                   | 43.50%    |   0.22 sec   | 36.29
action_prediction              | 32.00%    |   0.27 sec   | 50.84
episodic_reasoning             | 45.50%    |   0.38 sec   | 413.50
action_antonym                 | 75.00%    |   0.26 sec   | 33.18
action_count                   | 44.50%    |   0.31 sec   | 151.14
scene_transition               | 63.50%    |   0.42 sec   | 434.17
object_shuffle                 | 36.50%    |   0.30 sec   | 49.24
object_existence               | 59.50%    |   0.22 sec   | 36.52
fine_grained_pose              | NO DATA
unexpected_action              | 65.00%    |   0.28 